In [2]:
#Import Libraries
import requests
import os
import json
from datetime import datetime
from socrata import Socrata
from sodapy import Socrata as S1
from socrata.authorization import Authorization


In [88]:
# Environment variables or replace with direct credentials
domain = "data.colorado.gov"
username = os.getenv('SOCRATA_USERNAME')
password = os.getenv('SOCRATA_PASSWORD')
auth = (username, password)

In [89]:
s_auth = Authorization(domain, username, password)

In [114]:
# Dataset ID to modify
fxfs = ["r76i-uws6","iuxm-ddzz","ucnv-vw74"]

In [115]:
#Get the Target Catalog Only getting published datasets
domainurl = f"https://{domain}/api/catalog/v1?domains={domain}&only=dataset&published=true" 
#domainurl = f"https://{domain}/api/catalog/v1?domains={domain}&ids=r76i-uws6"
#make a http request for basic auth
response = requests.get(domainurl, auth=auth)
if response.status_code != 200:
    print("View Catalog failed! " + response.text)
    os._exit(1)
    tranformList = []

#uniqueColumns = []
catalogdata = json.loads(response.text)
catalogdata['results']

[{'resource': {'name': 'Census Zip Codes in Colorado 2018',
   'id': 'r76i-uws6',
   'parent_fxf': [],
   'description': '',
   'attribution': None,
   'attribution_link': None,
   'contact_email': None,
   'type': 'dataset',
   'updatedAt': '2024-09-10T13:55:55.000Z',
   'createdAt': '2024-09-10T13:53:19.000Z',
   'metadata_updated_at': '2024-09-10T13:55:55.000Z',
   'data_updated_at': '2024-09-10T13:55:43.000Z',
   'page_views': {'page_views_last_week': 0,
    'page_views_last_month': 0,
    'page_views_total': 0,
    'page_views_last_week_log': None,
    'page_views_last_month_log': None,
    'page_views_total_log': None},
   'columns_name': ['the_geom',
    'zip_code',
    'pop',
    'hispanic',
    'white_nh',
    'black_nh',
    'ntvam_nh',
    'asian_nh',
    'hawpi_nh',
    'other_nh',
    'twoplus_nh',
    'male',
    'female',
    'ageless5',
    'age5_9',
    'age10_14',
    'age15_19',
    'age20_24',
    'age25_29',
    'age30_34',
    'age35_39',
    'age40_44',
    'age4

In [1]:
transforms_dict = {}
for asset in catalogdata['results']:
    print(asset['resource']["id"])
    if asset['resource']["id"] in fxfs:
        fxf = asset['resource']["id"]
        cols = asset['resource']['columns_field_name']
        print(fxf)
        transforms_dict[fxf]=cols[1:]
        for col in cols[1:]:
            print(col)
        print('_'*100)
        
transforms_dict

NameError: name 'catalogdata' is not defined

In [117]:
for fxf,cols in transforms_dict.items():
    print(fxf,cols)

r76i-uws6 ['zip_code', 'pop', 'hispanic', 'white_nh', 'black_nh', 'ntvam_nh', 'asian_nh', 'hawpi_nh', 'other_nh', 'twoplus_nh', 'male', 'female', 'ageless5', 'age5_9', 'age10_14', 'age15_19', 'age20_24', 'age25_29', 'age30_34', 'age35_39', 'age40_44', 'age45_49', 'age50_54', 'age55_59', 'age60_64', 'age65_69', 'age70_74', 'age75_79', 'age80_84', 'age85pl', 'ageless18', 'age18_24', 'med_age', 'households', 'familyhh', 'nonfamhh', 'hhldralone', 'hhldr_naln', 'housing_un', 'occ_hu', 'vac_hu', 'owned', 'rented', 'pop25plus', 'nohsdipl', 'hsgrad_sc', 'bachl_hghr', 'med_hh_inc', 'med_fam_in', 'per_cap_in', 'med_yr_blt', 'med_c_rent', 'med_g_rent', 'med_hm_val', 'citz_birth', 'citz_nat', 'not_citz', 'born_in_co', 'brn_oth_st', 'ntv_b_o_us', 'foreign_b', 'pop_1p', 'same_house', 'same_cnty', 'same_state', 'diff_state', 'frm_abroad', 'wrkrs_16pl', 'car_all', 'car_alone', 'car_carpoo', 'public_trn', 'pt_bus', 'pt_other', 'bike', 'walk', 'tr_other', 'wrk_home', 'w_16pl_nh', 't_less_10', 't_10_19',

In [118]:
client = Socrata(s_auth)

# Extract the dataset ID from the dictionary instead of passing the dictionary directly
for fxf,cols in transforms_dict.items():
    print (fxf,cols)

    c_ds = client.views.lookup(fxf)
    revision = c_ds.revisions.create_replace_revision()
    source = revision.source_from_dataset()
    output_schema = source.get_latest_input_schema().get_latest_output_schema()
    new_output_schema = output_schema

# Assuming cols is a list of column names
    for f_name in cols:  
        print(f'Processing column: {f_name}')
#        c_ds = client.views.lookup(f_name)
    
    # Change the column datatype to number
        new_output_schema = new_output_schema.change_column_transform(f_name).to(f'to_number({f_name})')
        print(f'Changing Display Datatype to Number for column: {f_name}')

# Apply the new output schema
    new_output_schema.run()
    apply_response = revision.apply(output_schema=new_output_schema)
    apply_response.wait_for_finish(progress=lambda job: print())
print(f'{f_name} Dataset updated with new datatype')


r76i-uws6 ['zip_code', 'pop', 'hispanic', 'white_nh', 'black_nh', 'ntvam_nh', 'asian_nh', 'hawpi_nh', 'other_nh', 'twoplus_nh', 'male', 'female', 'ageless5', 'age5_9', 'age10_14', 'age15_19', 'age20_24', 'age25_29', 'age30_34', 'age35_39', 'age40_44', 'age45_49', 'age50_54', 'age55_59', 'age60_64', 'age65_69', 'age70_74', 'age75_79', 'age80_84', 'age85pl', 'ageless18', 'age18_24', 'med_age', 'households', 'familyhh', 'nonfamhh', 'hhldralone', 'hhldr_naln', 'housing_un', 'occ_hu', 'vac_hu', 'owned', 'rented', 'pop25plus', 'nohsdipl', 'hsgrad_sc', 'bachl_hghr', 'med_hh_inc', 'med_fam_in', 'per_cap_in', 'med_yr_blt', 'med_c_rent', 'med_g_rent', 'med_hm_val', 'citz_birth', 'citz_nat', 'not_citz', 'born_in_co', 'brn_oth_st', 'ntv_b_o_us', 'foreign_b', 'pop_1p', 'same_house', 'same_cnty', 'same_state', 'diff_state', 'frm_abroad', 'wrkrs_16pl', 'car_all', 'car_alone', 'car_carpoo', 'public_trn', 'pt_bus', 'pt_other', 'bike', 'walk', 'tr_other', 'wrk_home', 'w_16pl_nh', 't_less_10', 't_10_19',
































civ_ni_p Dataset updated with new datatype
